# CityTrack S02 Ablation R3 - occlusion

Slug: `ali369/citytrack-s02-occ`

Mode: CPU, cached Stage-1/2 -> stages 3-5

Commit pin: `paper-tests@b5aef3e`

S02 camera scope: `S02_c006`, `S02_c007`, `S02_c008`

Overrides: `stage4.association.occlusion_aware.enabled=true`

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

RUN_LABEL = "R3"
KERNEL_SLUG = "citytrack-s02-occ"
RUN_ID = "citytrack_s02_occ"
RUN_OVERRIDES = ["stage4.association.occlusion_aware.enabled=true"]
TARGET_CAMERAS = ["S02_c006", "S02_c007", "S02_c008"]
REPO_URL = "https://github.com/MRKDaGods/gp.git"
BRANCH = "paper-tests"
EXPECTED_COMMIT = "b5aef3e"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
DATA_OUT = Path("/tmp/pipeline_outputs")

if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT)])
try:
    subprocess.check_call(["git", "-C", str(PROJECT), "checkout", EXPECTED_COMMIT])
except subprocess.CalledProcessError:
    subprocess.check_call(["git", "-C", str(PROJECT), "fetch", "origin", EXPECTED_COMMIT, "--depth", "1"])
    subprocess.check_call(["git", "-C", str(PROJECT), "checkout", EXPECTED_COMMIT])
os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
head_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("git rev-parse HEAD:", head_sha)
if not head_sha.startswith(EXPECTED_COMMIT):
    raise RuntimeError(f"Expected {EXPECTED_COMMIT}, got {head_sha}")
print(f"Repo ready at {PROJECT}")

## Install CPU Dependencies

In [ ]:
def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


try:
    import faiss
    print(f"faiss ok ({faiss.__version__})")
except ImportError:
    pip_install("faiss-cpu")

try:
    import trackeval
    print("trackeval ok")
except ImportError:
    pip_install("git+https://github.com/JonathonLuiten/TrackEval.git")

pip_install("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click", "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=str(PROJECT))

FAILED = []
for label, module in [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("trackeval", "trackeval"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
]:
    try:
        __import__(module)
        print(f"  OK {label}")
    except ImportError as exc:
        print(f"  MISSING {label}: {exc}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED}")
print("CPU dependencies importable")

## Scope CityFlowV2 To S02

In [ ]:
for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount:16s} {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")

candidate_mounts = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = next((path for path in candidate_mounts if path.exists()), None)
if CITYFLOW_INPUT is None:
    raise FileNotFoundError("CityFlowV2 dataset not found; attach thanhnguyenle/data-aicity-2023-track-2")
print(f"CityFlowV2 input: {CITYFLOW_INPUT}")

TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT = PROJECT / "data" / "raw"
if DATA_RAW_PARENT.exists() or DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.is_symlink() or DATA_RAW_PARENT.is_file():
        DATA_RAW_PARENT.unlink()
    else:
        shutil.rmtree(DATA_RAW_PARENT)
DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT.symlink_to(TMP_DATA)

DATA_RAW = TMP_DATA / "cityflowv2"
if DATA_RAW.exists():
    shutil.rmtree(DATA_RAW)
DATA_RAW.mkdir(parents=True, exist_ok=True)

for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir() or scene_dir.name != "S02":
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"
            if flat_name not in TARGET_CAMERAS:
                continue
            flat_dir = DATA_RAW / flat_name
            if not flat_dir.exists():
                flat_dir.symlink_to(cam_dir)

present = sorted(path.name for path in DATA_RAW.iterdir() if path.is_dir())
print(f"S02 camera scope: {present}")
missing_cams = sorted(set(TARGET_CAMERAS) - set(present))
if missing_cams:
    raise FileNotFoundError(f"Missing S02 cameras in flattened dataset: {missing_cams}")

print("S02 GT/data symlinks ready for CPU evaluation")

## Reuse Cached Stage-1/2 Features

In [ ]:
import numpy as np

INPUT_ROOT = Path("/kaggle/input")


def find_input_dir(slug: str, owner_slug: str, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct
    owner, _, kernel = owner_slug.partition("/")
    nested = INPUT_ROOT / "notebooks" / owner / kernel
    if nested.exists():
        return nested
    lowered_slug = slug.lower()
    lowered_hints = tuple(str(hint).lower() for hint in hints)
    for path in list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []:
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or all(hint in name for hint in lowered_hints):
            return path
    for path in INPUT_ROOT.rglob("checkpoint.tar.gz") if INPUT_ROOT.exists() else []:
        parent_text = str(path.parent).lower()
        if lowered_slug in parent_text or all(hint in parent_text for hint in lowered_hints):
            return path.parent
    return direct


def resolve_extracted_checkpoint(root: Path) -> tuple[str, Path] | None:
    if not root.exists() or not root.is_dir():
        return None
    metadata_path = root / "run_metadata.json"
    if metadata_path.exists():
        previous_meta = json.loads(metadata_path.read_text(encoding="utf-8"))
        run_name = previous_meta["run_name"]
        return run_name, root / run_name
    if (root / "stage2").exists():
        return root.name, root
    run_dirs = [path for path in root.iterdir() if path.is_dir() and (path / "stage2").exists()]
    if len(run_dirs) == 1:
        return run_dirs[0].name, run_dirs[0]
    return None


def find_kernel_output_file(owner_slug: str, filename: str, hints=()) -> Path:
    slug = owner_slug.split("/", 1)[1]
    input_dir = find_input_dir(slug, owner_slug, hints=hints)
    direct = input_dir / filename
    if direct.exists():
        print(f"Mounted {owner_slug}: {input_dir}")
        return direct
    if resolve_extracted_checkpoint(input_dir) is not None:
        print(f"Mounted extracted {owner_slug}: {input_dir}")
        return input_dir
    if INPUT_ROOT.exists():
        for candidate in INPUT_ROOT.rglob(filename):
            parent_text = str(candidate.parent).lower()
            if slug.lower() in parent_text or all(str(hint).lower() in parent_text for hint in hints):
                print(f"Mounted {owner_slug}: {candidate.parent}")
                return candidate
        for marker in INPUT_ROOT.rglob("run_metadata.json"):
            parent_text = str(marker.parent).lower()
            if slug.lower() in parent_text or all(str(hint).lower() in parent_text for hint in hints):
                print(f"Mounted extracted {owner_slug}: {marker.parent}")
                return marker.parent
    print(f"{filename} not found at {direct}; trying Kaggle API fallback for {owner_slug}")
    dl_dir = Path("/tmp") / f"kaggle_{slug}_download"
    dl_dir.mkdir(parents=True, exist_ok=True)
    pattern = "^" + filename.replace(".", r"\.") + "$"
    result = subprocess.run(
        ["kaggle", "kernels", "output", owner_slug, "--file-pattern", pattern, "-p", str(dl_dir)],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    downloaded = dl_dir / filename
    if downloaded.exists() and downloaded.stat().st_size > 0:
        print(f"Downloaded {filename} from {owner_slug}")
        return downloaded
    visible = [str(path) for path in INPUT_ROOT.rglob(filename)] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(f"Could not resolve {filename} for {owner_slug}. Visible matches: {visible[:20]}")


def extract_checkpoint(checkpoint_path: Path, extract_dir: Path) -> tuple[str, Path]:
    mounted = resolve_extracted_checkpoint(checkpoint_path)
    if mounted is not None:
        print(f"Using extracted checkpoint directory: {checkpoint_path}")
        return mounted
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {checkpoint_path} ({checkpoint_path.stat().st_size / 1024**2:.1f} MB)")
    with tarfile.open(str(checkpoint_path), "r:gz") as tar:
        tar.extractall(str(extract_dir))
    extracted = resolve_extracted_checkpoint(extract_dir)
    if extracted is not None:
        return extracted
    raise FileNotFoundError(f"No run_metadata.json or unique stage2 run dir found in {extract_dir}")


SOURCE_14C = "yahiaakhalafallah/14c-tta-stage2"
SOURCE_10A = "yahiaakhalafallah/mtmc-10a-stages-0-2"
checkpoint_14c = find_kernel_output_file(SOURCE_14C, "checkpoint.tar.gz", hints=("14c", "tta", "stage2"))
SOURCE_14C_RUN_NAME, SOURCE_14C_RUN_DIR = extract_checkpoint(checkpoint_14c, Path("/tmp/14c_checkpoint"))
print(f"Loaded 14c run: {SOURCE_14C_RUN_NAME}")

SOURCE_10A_RUN_DIR = None
stage1_source = SOURCE_14C_RUN_DIR / "stage1"
if not list(stage1_source.glob("tracklets_*.json")):
    checkpoint_10a = find_kernel_output_file(SOURCE_10A, "checkpoint.tar.gz", hints=("10a", "stages", "0", "2"))
    SOURCE_10A_RUN_NAME, SOURCE_10A_RUN_DIR = extract_checkpoint(checkpoint_10a, Path("/tmp/10a_checkpoint"))
    print(f"Loaded 10a fallback run: {SOURCE_10A_RUN_NAME}")
    stage1_source = SOURCE_10A_RUN_DIR / "stage1"
stage2_source = SOURCE_14C_RUN_DIR / "stage2"
if not stage1_source.exists():
    raise FileNotFoundError(f"Stage1 tracklets not found: {stage1_source}")
if not stage2_source.exists():
    raise FileNotFoundError(f"Stage2 features not found: {stage2_source}")

RUN_DIR = DATA_OUT / RUN_ID
if RUN_DIR.exists() or RUN_DIR.is_symlink():
    if RUN_DIR.is_symlink() or RUN_DIR.is_file():
        RUN_DIR.unlink()
    else:
        shutil.rmtree(RUN_DIR)
(RUN_DIR / "stage1").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "stage2").mkdir(parents=True, exist_ok=True)

for camera_id in TARGET_CAMERAS:
    src = stage1_source / f"tracklets_{camera_id}.json"
    if not src.exists():
        raise FileNotFoundError(src)
    shutil.copy2(src, RUN_DIR / "stage1" / src.name)

index_map = json.loads((stage2_source / "embedding_index.json").read_text(encoding="utf-8"))
keep_indices = [idx for idx, item in enumerate(index_map) if item.get("camera_id") in TARGET_CAMERAS]
if not keep_indices:
    raise RuntimeError("No S02 rows found in cached Stage-2 embedding_index.json")
keep_set = set(keep_indices)
filtered_index = [item for idx, item in enumerate(index_map) if idx in keep_set]

def subset_npy(name: str, required: bool = False) -> None:
    src = stage2_source / name
    dst = RUN_DIR / "stage2" / name
    if not src.exists():
        if required:
            raise FileNotFoundError(src)
        print(f"Optional Stage-2 artifact absent: {src}")
        return
    arr = np.load(src)
    if arr.shape[0] != len(index_map):
        if required:
            raise ValueError(f"{src} rows {arr.shape[0]} != index_map rows {len(index_map)}")
        print(f"Skipping {name}: row mismatch {arr.shape[0]} != {len(index_map)}")
        return
    np.save(dst, arr[keep_indices].astype(arr.dtype, copy=False))
    print(f"Subset {name}: {arr.shape} -> {arr[keep_indices].shape}")

subset_npy("embeddings.npy", required=True)
subset_npy("hsv_features.npy", required=True)
subset_npy("embeddings_secondary.npy", required=False)
subset_npy("embeddings_tertiary.npy", required=False)
subset_npy("embeddings_quaternary.npy", required=False)
(RUN_DIR / "stage2" / "embedding_index.json").write_text(json.dumps(filtered_index, indent=2), encoding="utf-8")

mq_src = stage2_source / "multi_query_embeddings.npz"
if mq_src.exists():
    with np.load(mq_src) as data:
        if "embeddings" in data and data["embeddings"].shape[0] == len(index_map):
            np.savez_compressed(RUN_DIR / "stage2" / "multi_query_embeddings.npz", embeddings=data["embeddings"][keep_indices])
            print("Subset multi_query_embeddings.npz")

run_latest = DATA_OUT / "run_latest"
if run_latest.exists() or run_latest.is_symlink():
    if run_latest.is_symlink() or run_latest.is_file():
        run_latest.unlink()
    else:
        shutil.rmtree(run_latest)
run_latest.symlink_to(RUN_DIR, target_is_directory=True)

print(f"Prepared cached S02 artifacts in {RUN_DIR}")
print(f"Tracklet files: {[path.name for path in sorted((RUN_DIR / 'stage1').glob('tracklets_*.json'))]}")
print(f"Filtered feature rows: {len(filtered_index)}")

## Run Stages 3-5

In [ ]:
os.chdir(str(PROJECT))
base_overrides = [
    f"project.run_name={RUN_ID}",
    f"project.output_dir={DATA_OUT}",
    f"stage5.ground_truth_dir={DATA_RAW}",
    f"stage4.association.tertiary_embeddings.path={RUN_DIR / 'stage2' / 'embeddings_tertiary.npy'}",
]
all_overrides = base_overrides + RUN_OVERRIDES
cmd = [
    sys.executable,
    "scripts/run_pipeline.py",
    "--config",
    "configs/default.yaml",
    "--dataset-config",
    "configs/datasets/cityflowv2.yaml",
    "--stages",
    "3,4,5",
]
for item in all_overrides:
    cmd += ["--override", item]

manifest = {
    "run_label": RUN_LABEL,
    "kernel_slug": KERNEL_SLUG,
    "run_id": RUN_ID,
    "commit": EXPECTED_COMMIT,
    "target_cameras": TARGET_CAMERAS,
    "overrides": RUN_OVERRIDES,
    "all_overrides": all_overrides,
    "stages": "3,4,5",
    "mode": "cpu_cached_stage1_stage2_filtered_to_s02",
    "cached_sources": ["yahiaakhalafallah/14c-tta-stage2", "yahiaakhalafallah/mtmc-10a-stages-0-2"],
}
(WORK_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2))

print("CMD:", " ".join(str(part) for part in cmd))
print("=" * 80)
start = time.time()
result = subprocess.run(cmd, cwd=str(PROJECT))
elapsed_min = (time.time() - start) / 60.0
print("=" * 80)
if result.returncode != 0:
    raise SystemExit(result.returncode)
print(f"Stages 3-5 completed in {elapsed_min:.1f} min")

## Stage-5 Results

In [ ]:
stage5_dir = RUN_DIR / "stage5"
metrics_path = stage5_dir / "evaluation_report.json"
if not metrics_path.exists():
    raise FileNotFoundError(metrics_path)

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
details = metrics.get("details", {}) or {}
per_camera = details.get("per_camera", {}) or {}
mtmc_idf1 = metrics.get("mtmc_idf1") or details.get("mtmc_idf1") or metrics.get("idf1")

summary = {
    "run_label": RUN_LABEL,
    "kernel_slug": KERNEL_SLUG,
    "run_id": RUN_ID,
    "commit": EXPECTED_COMMIT,
    "target_cameras": TARGET_CAMERAS,
    "overrides": RUN_OVERRIDES,
    "mtmc_idf1": mtmc_idf1,
    "idf1": metrics.get("idf1"),
    "mota": metrics.get("mota"),
    "hota": metrics.get("hota"),
    "id_switches": metrics.get("id_switches"),
    "details_mtmc_idf1": details.get("mtmc_idf1"),
    "details_mtmc_mota": details.get("mtmc_mota"),
    "details_mtmc_id_switches": details.get("mtmc_id_switches"),
    "per_camera": per_camera,
    "s02_c006": per_camera.get("S02_c006"),
    "metrics_path": str(metrics_path),
}

print("S02 MTMC IDF1:", mtmc_idf1)
print("Per-camera metrics:")
for camera_id in TARGET_CAMERAS:
    print(f"  {camera_id}: {per_camera.get(camera_id)}")

summary_path = WORK_DIR / f"{KERNEL_SLUG}_results.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
shutil.copy2(metrics_path, WORK_DIR / f"{KERNEL_SLUG}_evaluation_report.json")
html_report = stage5_dir / "evaluation_report.html"
if html_report.exists():
    shutil.copy2(html_report, WORK_DIR / f"{KERNEL_SLUG}_evaluation_report.html")

stage5_tar = WORK_DIR / f"{KERNEL_SLUG}_stage5.tar.gz"
with tarfile.open(str(stage5_tar), "w:gz") as tar:
    tar.add(str(stage5_dir), arcname="stage5")
print(f"Wrote {summary_path}")
print(f"Packed Stage-5 outputs: {stage5_tar}")